In [1]:
# ==============================================================================
# 🛡️ HÜCRE 1: ÇEKİRDEK ORTAM, ZIRHLAMA, FAST-REID VE BAĞIMLILIKLAR
# ==============================================================================
import os, sys, gc, subprocess, glob
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

print("📦 1. Kütüphaneler kuruluyor...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "opencv-python", "matplotlib", "scikit-image", "einops", "kornia",
                "timm", "yacs", "joblib", "natsort", "h5py", "tqdm", "ptflops",
                "seaborn", "addict", "future", "lmdb", "numpy", "pyyaml", "requests",
                "scipy", "yapf", "lpips", "cython", "cython_bbox", "pandas",
                "xmltodict", "loguru", "gdown", "lapx", "motmetrics", "filterpy",
                "thop", "faiss-cpu", "faiss-gpu", "tabulate"])

print("📥 2. Repolar klonlanıyor (DeepRFT, LightStab, HybridSORT)...")
for repo, url in [("DeepRFT", "https://github.com/INVOKERer/DeepRFT.git -b AAAI2023"),
                  ("LightStab", "https://github.com/liutao23/LightStab.git"),
                  ("HybridSORT", "https://github.com/ymzis69/HybridSORT.git")]:
    if not os.path.exists(f'/content/{repo}'):
        os.system(f"git clone {url} /content/{repo}")

print("🛠️ 3. Sistem yamaları (NumPy 2.x & Headless Matplotlib) uygulanıyor...")
# LightStab Headless Yama
ls_file = "/content/LightStab/model/LightMotionEsitimation.py"
if os.path.exists(ls_file):
    with open(ls_file, "r") as f: c = f.read()
    with open(ls_file, "w") as f: f.write(c.replace("matplotlib.use('TkAgg')", "matplotlib.use('Agg')"))

# HybridSORT NumPy 2.x Yaması
for py_file in glob.glob("/content/HybridSORT/**/*.py", recursive=True):
    try:
        with open(py_file, 'r', encoding='utf-8') as f: code = f.read()
        for old, new in [("np.float(", "float("), ("np.int(", "int("), ("np.bool(", "bool("),
                         ("astype(float32)", "astype(np.float32)"), ("dtype=float32", "dtype=np.float32"),
                         ("astype(int32)", "astype(np.int32)")]:
            code = code.replace(old, new)
        with open(py_file, 'w', encoding='utf-8') as f: f.write(code)
    except: pass

os.chdir("/content/HybridSORT")
if not os.path.exists("yolox.egg-info"):
    os.system("pip install -e . --no-build-isolation --no-deps -q")

print("🧬 3.5. FastReID Modülü Aktif Ediliyor...")
fast_reid_req = "/content/HybridSORT/fast_reid/docs/requirements.txt"
if os.path.exists(fast_reid_req):
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", fast_reid_req, "-q"])

os.chdir("/content")
print("✅ Hücre 1 Tamamlandı: Ortam %100 Hazır, Re-ID Destekli ve Zırhlı.")

Mounted at /content/drive
📦 1. Kütüphaneler kuruluyor...
📥 2. Repolar klonlanıyor (DeepRFT, LightStab, HybridSORT)...
🛠️ 3. Sistem yamaları (NumPy 2.x & Headless Matplotlib) uygulanıyor...
🧬 3.5. FastReID Modülü Aktif Ediliyor...
✅ Hücre 1 Tamamlandı: Ortam %100 Hazır, Re-ID Destekli ve Zırhlı.


In [2]:
# ==============================================================================
# 📂 HÜCRE 2: VERİSETİ BAĞLANTISI (MOT17-04) VE İLKLEME
# ==============================================================================
import os
import glob
import cv2
import torch
import shutil

# 1. Veriseti Yolları (Drive'a yeni yüklediğin MOT17-04 dizini)
dataset_base = "/content/drive/MyDrive/Spikedge_Staj/Tracking/MOT17_Dataset/train/MOT17-04-FRCNN"
img_dir = os.path.join(dataset_base, "img1")
gt_path = os.path.join(dataset_base, "gt/gt.txt")

image_files = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))

print("="*50)
print(f"📁 Seçilen Senaryo: MOT17-04-FRCNN")
print(f"🎞️ Toplam Kare Sayısı: {len(image_files)}")
print(f"📊 Ground Truth Konumu: {gt_path}")
print("="*50)

if not image_files:
    raise FileNotFoundError("❌ Görüntüler bulunamadı! Drive yolunu kontrol edin.")
if not os.path.exists(gt_path):
    raise FileNotFoundError("❌ gt.txt bulunamadı! Verisetinin eksiksiz yüklendiğinden emin olun.")

# 2. YOLOX Ağırlık Yükleme ve Zırhlı Taşıma
drive_yolox = "/content/drive/MyDrive/Spikedge_Staj/Tracking/pretrained/ocsort_x_mot17.pth.tar"
local_yolox = "/content/HybridSORT/pretrained/ocsort_x_mot17.pth.tar"

os.makedirs(os.path.dirname(local_yolox), exist_ok=True)
if os.path.exists(drive_yolox) and not os.path.exists(local_yolox):
    shutil.copy(drive_yolox, local_yolox)
    print("📥 YOLOX modeli Drive'dan çekildi.")
elif not os.path.exists(local_yolox):
    raise FileNotFoundError(f"❌ {drive_yolox} konumunda model bulunamadı!")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Hücre 2 Tamamlandı: Veriseti dizinlendi. Çalışma birimi: {device.type.upper()}")

📁 Seçilen Senaryo: MOT17-04-FRCNN
🎞️ Toplam Kare Sayısı: 1050
📊 Ground Truth Konumu: /content/drive/MyDrive/Spikedge_Staj/Tracking/MOT17_Dataset/train/MOT17-04-FRCNN/gt/gt.txt
📥 YOLOX modeli Drive'dan çekildi.
✅ Hücre 2 Tamamlandı: Veriseti dizinlendi. Çalışma birimi: CUDA


In [3]:
# ==============================================================================
# 🛡️ YENİ HÜCRE 2.5: DEEPRFT ORİJİNAL COMMIT ENTEGRASYONU (DEC 5, 2021)
# ==============================================================================
import os
import sys
import subprocess
import importlib.util
import shutil

print("⚡ [Sistem] DeepRFT orijinal commit (Dec 5, 2021) yapılandırılıyor...", flush=True)

subprocess.run([sys.executable, "-m", "pip", "install", "einops", "timm", "kornia", "ptflops", "natsort", "yacs", "joblib", "addict", "future", "yapf", "lpips", "-q"])

deeprft_dir = "/content/DeepRFT"
if os.path.exists(deeprft_dir):
    shutil.rmtree(deeprft_dir)

print(" 📥 DeepRFT reposu klonlanıyor...", flush=True)
subprocess.run(["git", "clone", "https://github.com/INVOKERer/DeepRFT.git", deeprft_dir])

# 🛠️ Ekran görüntüsündeki orijinal commit'e git!
os.chdir(deeprft_dir)
# Ekran görüntüsünün commit hash'i veya en yakın kararlı sürüm
subprocess.run(["git", "checkout", "a88bf49"])
os.chdir("/content")

if deeprft_dir not in sys.path:
    sys.path.insert(0, deeprft_dir)

try:
    file_path = os.path.join(deeprft_dir, "DeepRFT_MIMO.py")
    if not os.path.exists(file_path):
        raise Exception("DeepRFT_MIMO.py dosyası commit içinde bulunamadı!")

    print(" 🔍 DeepRFT_MIMO.py modülü fiziksel olarak RAM'e alınıyor...", flush=True)
    spec = importlib.util.spec_from_file_location("DeepRFT_MIMO", file_path)
    DeepRFT_MIMO = importlib.util.module_from_spec(spec)
    sys.modules["DeepRFT_MIMO"] = DeepRFT_MIMO
    spec.loader.exec_module(DeepRFT_MIMO)

    import inspect
    DeepRFT_bulundu = None
    for isim, obj in inspect.getmembers(DeepRFT_MIMO, inspect.isclass):
        if 'DeepRFT' in isim:
            DeepRFT_bulundu = obj
            print(f" 🎯 Nokta Atışı Başarılı! Sınıf bulundu: {isim}", flush=True)
            break

    if DeepRFT_bulundu is None:
        raise Exception("DeepRFT sınıfı bulunamadı!")

    import __main__
    __main__.DeepRFT = DeepRFT_bulundu
    print(" 🟢 [BAŞARILI] Orijinal SOTA model Python çekirdeğine MÜHÜRLENDİ!", flush=True)

except Exception as e:
    print(f" ❌ [Kritik Sistem Hatası] {e}", flush=True)

⚡ [Sistem] DeepRFT orijinal commit (Dec 5, 2021) yapılandırılıyor...
 📥 DeepRFT reposu klonlanıyor...
 🔍 DeepRFT_MIMO.py modülü fiziksel olarak RAM'e alınıyor...
 🎯 Nokta Atışı Başarılı! Sınıf bulundu: DeepRFT
 🟢 [BAŞARILI] Orijinal SOTA model Python çekirdeğine MÜHÜRLENDİ!


In [4]:
# ==============================================================================
# ✈️ PRE-FLIGHT CHECK (Senkronizasyon Korumalı)
# ==============================================================================
import torch
import os
import time

weights_path = "/content/drive/MyDrive/Spikedge_Staj/Deblurring/model_GoPro.pth"

print("🔍 [Pre-Flight Check] Drive senkronizasyonu bekleniyor...", flush=True)

# Drive senkronizasyonu için kısa bir bekleme ve kontrol döngüsü
for i in range(10):
    if os.path.exists(weights_path):
        print(f" ✅ Dosya algılandı! Boyut: {os.path.getsize(weights_path) / (1024*1024):.2f} MB", flush=True)
        break
    time.sleep(2)

try:
    import __main__
    if not hasattr(__main__, 'DeepRFT'):
        raise Exception("DeepRFT modeli RAM'de bulunamadı! Lütfen önce Hücre 2.5'i çalıştırın.")

    test_model = __main__.DeepRFT()

    if not os.path.exists(weights_path):
        raise Exception(f"Ağırlık dosyası hâl่ะ senkronize olmadı: {weights_path}")

    checkpoint = torch.load(weights_path, map_location="cpu", weights_only=False)
    raw_state_dict = checkpoint.get("state_dict", checkpoint.get("model", checkpoint))

    clean_state_dict = {k.replace('module.', ''): v for k, v in raw_state_dict.items()}

    test_model.load_state_dict(clean_state_dict, strict=True)

    print("\n" + "🟢"*30, flush=True)
    print(" 🎉 MÜKEMMEL! Altın madeni ağırlık dosyası model mimarisiyle %100 UYUMLU!", flush=True)
    print("🟢"*30, flush=True)

except Exception as e:
    print(f"\n❌ [HATA]:\n{e}", flush=True)

🔍 [Pre-Flight Check] Drive senkronizasyonu bekleniyor...
 ✅ Dosya algılandı! Boyut: 41.51 MB

🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢
 🎉 MÜKEMMEL! Altın madeni ağırlık dosyası model mimarisiyle %100 UYUMLU!
🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢


In [5]:
# =============================================================================
# 🚀 HÜCRE 3: ULTIMATE ENTERPRISE PIPELINE V40 (ZERO-PIXEL ARMOR + SOTA RE-ID)
# =============================================================================
import os
import time
import cv2
import glob
import numpy as np
import torch
import gc
import sys
import subprocess
import shutil
import configparser
import re

print("🛡️ [Ultimate Enterprise Pipeline V40] Sıfır Piksel Zırhı, Fast-ReID ve ECC Aktif...", flush=True)

FORCE_CLEAN_RUN = False
subprocess.run([sys.executable, "-m", "pip", "install", "thop", "loguru", "lap", "cython_bbox", "faiss-gpu", "filterpy", "scipy", "-q"])

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
cv2.setNumThreads(1)

def aggressive_ram_purge():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def get_sequence_fps(seq_base_path):
    ini_path = os.path.join(seq_base_path, "seqinfo.ini")
    if os.path.exists(ini_path):
        config = configparser.ConfigParser()
        config.read(ini_path)
        try: return float(config['Sequence']['frameRate'])
        except Exception: pass
    return 30.0

def check_video_health(path):
    if not os.path.exists(path) or os.path.getsize(path) < 10000: return False
    cap = cv2.VideoCapture(path)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return frames > 10

def safe_drive_mirror(local_path, drive_path, stage_name):
    print(f" 🎬 [{stage_name}] H.264 formatında mühürleniyor...", flush=True)
    os.makedirs(os.path.dirname(drive_path), exist_ok=True)
    cmd = f"ffmpeg -y -i '{local_path}' -c:v libx264 -pix_fmt yuv420p -preset fast -crf 17 '{drive_path}'"
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if res.returncode != 0 or not os.path.exists(drive_path) or not check_video_health(drive_path):
        shutil.copy(local_path, drive_path)
    time.sleep(1)
    if check_video_health(drive_path):
        size_mb = os.path.getsize(drive_path) / (1024 * 1024)
        print(f" ✅ [{stage_name}] Başarılı! ({size_mb:.2f} MB)", flush=True)

# =============================================================================
# 🎛️ MODÜLLER: DEEP RFT (GOPRO) İLE KUSURSUZ PİKSELLER
# =============================================================================
def load_deblur_model(weights_path: str, device: str = "cuda"):
    device = torch.device(device if torch.cuda.is_available() else "cpu")
    PROJECT_ROOT = "/content/drive/MyDrive/Spikedge_Staj"
    if not os.path.exists(weights_path) and not os.path.isabs(weights_path):
        weights_path = os.path.join(PROJECT_ROOT, "Deblurring", weights_path)
    try:
        import __main__
        model = __main__.DeepRFT()
        if os.path.exists(weights_path):
            checkpoint = torch.load(weights_path, map_location=device, weights_only=False)
            state_dict = checkpoint.get("state_dict", checkpoint.get("model", checkpoint))
            clean_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
            model.load_state_dict(clean_state_dict, strict=True)
        model.to(device); model.eval()
        return model
    except Exception:
        return None

def run_deblurring(frames: list, model, device: str = "cuda") -> list:
    frames_arr = np.array(frames)
    deblurred = []
    if model is None:
        for img in frames_arr:
            gaussian = cv2.GaussianBlur(img, (0, 0), 2.0)
            sharpened = cv2.addWeighted(img, 1.5, gaussian, -0.5, 0)
            deblurred.append(np.clip(sharpened, 0, 255).astype(np.uint8))
        return deblurred

    device = torch.device(device if torch.cuda.is_available() else "cpu")
    for img in frames_arr:
        inp = torch.from_numpy(img).float().permute(2, 0, 1).unsqueeze(0) / 255.0
        inp = inp.to(device)
        with torch.no_grad():
            out = model(inp)
            if isinstance(out, (list, tuple)): out = out[0]
        out = torch.clamp(out, 0.0, 1.0)
        out_np = (out.squeeze(0).permute(1, 2, 0).cpu().numpy() * 255.0).astype(np.uint8)
        gaussian = cv2.GaussianBlur(out_np, (0, 0), 1.0)
        deblurred.append(cv2.addWeighted(out_np, 1.3, gaussian, -0.3, 0))
    return deblurred

# =============================================================================
# 🧹 SIFIRDAN TEMİZ KURULUM VE GÜVENLİ YAMALAR
# =============================================================================
os.chdir("/content")
hybris_dir = "/content/HybridSORT"

if os.path.exists(hybris_dir):
    print(" 🧹 Eski hatalı kurulum tamamen siliniyor...", flush=True)
    shutil.rmtree(hybris_dir)
    time.sleep(2)

print(" 📥 HybridSORT orijinal reposu klonlanıyor...", flush=True)
subprocess.run(["git", "clone", "https://github.com/ymzis69/HybridSORT.git", hybris_dir], capture_output=True, text=True)
os.chdir(hybris_dir)

print(" 🛠️ Zırhlı PyTorch ve Numpy 2.x Yamaları uygulanıyor...", flush=True)
for py_f in glob.glob("**/*.py", recursive=True):
    try:
        with open(py_f, 'r', encoding='utf-8') as f: content = f.read()

        content = content.replace("from collections import Mapping, OrderedDict", "from collections.abc import Mapping\nfrom collections import OrderedDict")
        content = content.replace("from collections import Mapping", "from collections.abc import Mapping")
        content = content.replace("from collections import Iterable", "from collections.abc import Iterable")
        content = content.replace("from torch._six import string_classes", "string_classes = (str, bytes)")
        content = content.replace("from torch._six import int_classes", "int_classes = int")
        content = content.replace('torch.load(ckpt_file, map_location="cpu")', 'torch.load(ckpt_file, map_location="cpu", weights_only=False)')
        content = content.replace("torch.load(ckpt_file, map_location='cpu')", "torch.load(ckpt_file, map_location='cpu', weights_only=False)")
        content = content.replace('torch.load(f, map_location=torch.device("cpu"))', 'torch.load(f, map_location=torch.device("cpu"), weights_only=False)')

        if "trk[:] =" in content:
            content = re.sub(
                r'trk\[:\]\s*=\s*\[pos\[0\]\[0\],\s*pos\[0\]\[1\],\s*pos\[0\]\[2\],\s*pos\[0\]\[3\],\s*kalman_score,\s*simple_score\[0\]\]',
                'p_flat = np.atleast_1d(pos).flatten(); s_flat = np.atleast_1d(simple_score).flatten(); trk[:] = [float(p_flat[0]), float(p_flat[1]), float(p_flat[2]), float(p_flat[3]), float(kalman_score), float(s_flat[0])]',
                content
            )
            content = re.sub(
                r'trk\[:\]\s*=\s*\[pos\[0\]\[0\],\s*pos\[0\]\[1\],\s*pos\[0\]\[2\],\s*pos\[0\]\[3\],\s*kalman_score,\s*simple_score\]',
                'p_flat = np.atleast_1d(pos).flatten(); s_flat = np.atleast_1d(simple_score).flatten(); trk[:] = [float(p_flat[0]), float(p_flat[1]), float(p_flat[2]), float(p_flat[3]), float(kalman_score), float(s_flat[0])]',
                content
            )
        with open(py_f, 'w', encoding='utf-8') as f: f.write(content)
    except Exception: pass

# =============================================================================
# 🚀 V40 KÖK NEDEN ÇÖZÜMÜ: FAST-REID "ZERO-PIXEL" (BOŞ MATRİS) ZIRHI
# =============================================================================
print(" 🛡️ Fast-ReID motoruna 'Sıfır Piksel (Empty Array)' zırhı entegre ediliyor...", flush=True)
fr_file = "fast_reid/fast_reid_interfece.py"
if os.path.exists(fr_file):
    with open(fr_file, 'r') as f:
        fr_code = f.read()

    if "import numpy as np" not in fr_code:
        fr_code = "import numpy as np\n" + fr_code

    regex_pattern = r"patch\s*=\s*cv2\.resize\(patch,\s*tuple\(self\.cfg\.INPUT\.SIZE_TEST\[::-1\]\),\s*interpolation=cv2\.INTER_LINEAR\).*"
    safe_patch = """if patch.size == 0 or patch.shape[0] == 0 or patch.shape[1] == 0:
                patch = np.zeros((self.cfg.INPUT.SIZE_TEST[0], self.cfg.INPUT.SIZE_TEST[1], 3), dtype=np.uint8)
            else:
                patch = cv2.resize(patch, tuple(self.cfg.INPUT.SIZE_TEST[::-1]), interpolation=cv2.INTER_LINEAR)"""

    fr_code = re.sub(regex_pattern, safe_patch, fr_code)

    with open(fr_file, 'w') as f:
        f.write(fr_code)
    print(" ✅ Fast-ReID motoru artık 'Hayalet Kutularda' bile çökmeyecek!")

print(" 🔬 demo_track.py otonom RegEx motoruyla zırhlanıyor...", flush=True)
demo_track_file = "tools/demo_track.py"
if os.path.exists(demo_track_file):
    with open(demo_track_file, "r") as f:
        code = f.read()

    code = code.replace("import argparse", "import argparse\nfrom fast_reid.fast_reid_interfece import FastReIDInterface\nimport numpy as np\nGLOBAL_ENCODER = None\n")

    inf_regex = re.compile(r'^([ \t]*)outputs,\s*img_info\s*=\s*predictor\.inference\(frame,\s*timer\)', re.MULTILINE)
    def repl_inf(match):
        indent = match.group(1)
        return (f"{indent}inf_res = predictor.inference(frame, timer)\n"
                f"{indent}outputs, img_info = inf_res[0], inf_res[1]\n"
                f"{indent}if outputs[0] is None:\n"
                f"{indent}    continue")
    code = inf_regex.sub(repl_inf, code)

    update_regex = re.compile(r'^([ \t]*)online_targets = tracker\.update\(outputs\[0\], \[img_info\[\'height\'\], img_info\[\'width\'\]\], exp\.test_size\)', re.MULTILINE)
    def repl_update(match):
        indent = match.group(1)
        return (f"{indent}global GLOBAL_ENCODER\n"
                f"{indent}if getattr(args, 'with_fastreid', False) and GLOBAL_ENCODER is None:\n"
                f"{indent}    GLOBAL_ENCODER = FastReIDInterface(args.fast_reid_config, args.fast_reid_weights, args.device)\n"
                f"{indent}id_feature = None\n"
                f"{indent}if GLOBAL_ENCODER is not None and outputs[0] is not None:\n"
                f"{indent}    bboxes_np = outputs[0][:, 0:4].cpu().numpy()\n"
                f"{indent}    img_h, img_w = img_info['raw_img'].shape[:2]\n"
                f"{indent}    bboxes_np[:, 0] = np.clip(bboxes_np[:, 0], 0, img_w - 1)\n"
                f"{indent}    bboxes_np[:, 1] = np.clip(bboxes_np[:, 1], 0, img_h - 1)\n"
                f"{indent}    bboxes_np[:, 2] = np.clip(bboxes_np[:, 2], 0, img_w - 1)\n"
                f"{indent}    bboxes_np[:, 3] = np.clip(bboxes_np[:, 3], 0, img_h - 1)\n"
                f"{indent}    bboxes_np[:, 2] = np.maximum(bboxes_np[:, 2], bboxes_np[:, 0] + 1)\n"
                f"{indent}    bboxes_np[:, 3] = np.maximum(bboxes_np[:, 3], bboxes_np[:, 1] + 1)\n"
                f"{indent}    id_feature = GLOBAL_ENCODER.inference(img_info['raw_img'], bboxes_np)\n"
                f"{indent}online_targets = tracker.update(outputs[0], [img_info['height'], img_info['width']], exp.test_size, id_feature=id_feature)")
    code = update_regex.sub(repl_update, code)

    code = code.replace("ch = cv2.waitKey(1)", "ch = -1")
    code = code.replace("cv2.waitKey(1)", "pass")
    code = code.replace("cv2.imshow(", "# cv2.imshow(")

    with open(demo_track_file, "w") as f:
        f.write(code)

# =============================================================================
# 🎯 V40 SOTA HİPERPARAMETRE OPTİMİZASYONU
# =============================================================================
print("\n🧠 Hiperparametreler Resmi SOTA ve ECC Kapasitesine Yükseltiliyor...", flush=True)
exp_file = "exps/example/mot/yolox_x_mix_det_hybrid_sort.py"
if os.path.exists(exp_file):
    with open(exp_file, 'r') as f: c_data = f.read()

    c_data = re.sub(r'self\.track_thresh\s*=\s*[\d\.]+', 'self.track_thresh = 0.6', c_data)
    c_data = re.sub(r'self\.nms_thresh\s*=\s*[\d\.]+', 'self.nms_thresh = 0.7', c_data)
    c_data = re.sub(r'self\.alpha\s*=\s*[\d\.]+', 'self.alpha = 0.8', c_data)

    c_data = re.sub(r'self\.hybrid_sort_with_reid\s*=\s*(True|False)', 'self.hybrid_sort_with_reid = True', c_data)
    c_data = re.sub(r'self\.with_fastreid\s*=\s*(True|False)', 'self.with_fastreid = True', c_data)

    with open(exp_file, 'w') as f: f.write(c_data)
    print(" ✅ Track_Thresh: 0.6 | NMS_Thresh: 0.7 | Alpha: 0.8 | ReID: True olarak kilitlendi!", flush=True)

# =============================================================================
# 🔄 ÇİFT SET DİNAMİK TOPLU İŞLEM DÖNGÜSÜ (TRAIN + TEST)
# =============================================================================
PROJECT_ROOT = "/content/drive/MyDrive/Spikedge_Staj"
LOCAL_DIR = "/content/local_processing"
os.makedirs(LOCAL_DIR, exist_ok=True)

all_sequences = [
    ("MOT17-02-FRCNN", "train"), ("MOT17-04-FRCNN", "train"), ("MOT17-05-FRCNN", "train"),
    ("MOT17-09-FRCNN", "train"), ("MOT17-10-FRCNN", "train"), ("MOT17-11-FRCNN", "train"), ("MOT17-13-FRCNN", "train"),
    ("MOT17-01-FRCNN", "test"),  ("MOT17-03-FRCNN", "test"),  ("MOT17-06-FRCNN", "test"),
    ("MOT17-07-FRCNN", "test"),  ("MOT17-08-FRCNN", "test"),  ("MOT17-12-FRCNN", "test"), ("MOT17-14-FRCNN", "test")
]

local_ckpt = os.path.join(hybris_dir, "pretrained/ocsort_x_mot17.pth.tar")
drive_ckpt = os.path.join(PROJECT_ROOT, "Tracking/pretrained/ocsort_x_mot17.pth.tar")
os.makedirs(os.path.dirname(local_ckpt), exist_ok=True)

if os.path.exists(drive_ckpt):
    shutil.copy(drive_ckpt, local_ckpt)
else:
    local_ckpt = os.path.join(hybris_dir, "weights/yolox_x.pth")
    os.makedirs(os.path.dirname(local_ckpt), exist_ok=True)
    os.system(f"curl -L -# -o {local_ckpt} https://github.com/ifzhang/ByteTrack/releases/download/v0.1_supp/yolox_x.pth")

reid_ckpt_local = os.path.join(hybris_dir, "pretrained/mot17_sbs_S50.pth")
reid_ckpt_drive = os.path.join(PROJECT_ROOT, "Tracking/pretrained/mot17_sbs_S50.pth")
if os.path.exists(reid_ckpt_drive): shutil.copy(reid_ckpt_drive, reid_ckpt_local)

for seq_name, split in all_sequences:
    print(f"\n" + "═"*70, flush=True)
    print(f"🎯 İŞLENEN SEKANS: {seq_name} ({split.upper()} SET) [V40 GOD MODE SOTA]", flush=True)
    print(f"═"*70, flush=True)

    DRIVE_INPUT_VIDEOS = os.path.join(PROJECT_ROOT, f"Tracking/input_videos_{split}")
    DRIVE_INTERMEDIATE = os.path.join(PROJECT_ROOT, f"Tracking/intermediate_{split}_reid")
    DRIVE_OUTPUT = os.path.join(PROJECT_ROOT, f"Tracking/output_tracks_{split}_reid")
    DRIVE_TELEMETRY = os.path.join(PROJECT_ROOT, f"Tracking/telemetry_{split}_reid")
    for d in [DRIVE_INPUT_VIDEOS, DRIVE_INTERMEDIATE, DRIVE_OUTPUT, DRIVE_TELEMETRY]: os.makedirs(d, exist_ok=True)

    found_paths = glob.glob(f"/content/drive/MyDrive/**/{seq_name}/img1", recursive=True)
    if not found_paths: found_paths = glob.glob(f"/content/drive/MyDrive/**/{seq_name}", recursive=True)

    if found_paths and os.path.exists(os.path.join(found_paths[0], "img1")): seq_base_path = found_paths[0]
    elif found_paths: seq_base_path = os.path.dirname(found_paths[0])
    else: continue

    seq_fps = get_sequence_fps(seq_base_path)
    image_files = sorted(glob.glob(os.path.join(seq_base_path, "img1", "*.jpg")))
    if not image_files: continue

    temp_source = os.path.join(LOCAL_DIR, f"{seq_name}_source.mp4")
    D_INPUT = os.path.join(DRIVE_INPUT_VIDEOS, f"{seq_name}_raw_input.mp4")

    L_STAGE1 = os.path.join(LOCAL_DIR, f"stage1_deblurred_{seq_name}.mp4")
    D_STAGE1 = os.path.join(DRIVE_INTERMEDIATE, f"stage1_deblurred_{seq_name}.mp4")
    D_STAGE3 = os.path.join(DRIVE_OUTPUT, f"final_tracked_V40_SOTA_{seq_name}.mp4")
    final_telemetry_path = os.path.join(DRIVE_TELEMETRY, f"{seq_name}.txt")

    if not FORCE_CLEAN_RUN and check_video_health(D_INPUT):
        if not os.path.exists(temp_source): shutil.copy(D_INPUT, temp_source)
    else:
        sample_img = cv2.imread(image_files[0])
        writer = cv2.VideoWriter(temp_source, cv2.VideoWriter_fourcc(*'mp4v'), seq_fps, (sample_img.shape[1], sample_img.shape[0]))
        for img_path in image_files: writer.write(cv2.imread(img_path))
        writer.release()
        safe_drive_mirror(temp_source, D_INPUT, f"Raw Input ({seq_name})")

    if not FORCE_CLEAN_RUN and check_video_health(D_STAGE1):
        if not os.path.exists(L_STAGE1): shutil.copy(D_STAGE1, L_STAGE1)
        print(" ⏭️ [Stage 1] DeepRFT Netleştirilmiş video bulundu, saniyeler içinde atlanıyor.", flush=True)
    else:
        print(f" 🚀 [Stage 1] DeepRFT (GoPro) Başlatılıyor...", flush=True)
        m1 = load_deblur_model("model_GoPro.pth", device="cuda")
        cap = cv2.VideoCapture(temp_source)
        writer_s1 = cv2.VideoWriter(L_STAGE1, cv2.VideoWriter_fourcc(*'mp4v'), seq_fps, (int(cap.get(3)), int(cap.get(4))))
        chunk = []
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            chunk.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if len(chunk) >= 100:
                for f in run_deblurring(chunk, m1, device="cuda"): writer_s1.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
                chunk.clear(); aggressive_ram_purge()
        if chunk:
            for f in run_deblurring(chunk, m1, device="cuda"): writer_s1.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
        cap.release(); writer_s1.release()
        try: del m1
        except: pass
        safe_drive_mirror(L_STAGE1, D_STAGE1, f"Stage 1 ({seq_name})")

    print(f" 🚀 [Stage 3] Hybrid-SORT-ReID (V40 SOTA) Doğrudan Stage 1 Üzerinde Çalışıyor...", flush=True)
    sota_out_dir = f"/content/sota_run_out_{seq_name}"
    os.makedirs(sota_out_dir, exist_ok=True)

    cmd = [sys.executable, f"{hybris_dir}/tools/demo_track.py", "--demo_type", "video", "-f", exp_file, "-c", local_ckpt,
           "--path", L_STAGE1, "--output_dir", sota_out_dir, "--device", "gpu", "--fp16", "--fuse", "--save_result",
           "--ECC", "--hybrid_sort_with_reid", "--with_fastreid", "--fast_reid_config", "fast_reid/configs/MOT17/sbs_S50.yml", "--fast_reid_weights", reid_ckpt_local]

    env = os.environ.copy(); env["PYTHONPATH"] = hybris_dir
    process = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

    log_history = []
    for line in process.stdout:
        log_history.append(line)
        if "Processing frame" in line or "save results to" in line: sys.stdout.write(line); sys.stdout.flush()
    process.wait()

    if process.returncode != 0:
        print(f"\n❌ [KRİTİK HATA] {seq_name} sekansında V40 SOTA motoru ÇÖKTÜ!", flush=True)
        print("".join(log_history[-50:]), flush=True)
        continue

    txt_files = glob.glob(os.path.join(sota_out_dir, "**/track_vis/*.txt"), recursive=True) + glob.glob(f"{hybris_dir}/YOLOX_outputs/**/track_vis/*.txt", recursive=True)
    if txt_files:
        latest_txt = max(txt_files, key=os.path.getmtime)
        shutil.copy(latest_txt, final_telemetry_path)
        print(f" 💾 [Telemetri] Saf SOTA Koordinatları Drive'a kilitlendi: {seq_name}.txt", flush=True)
    else:
        print(" ❌ [Hata] Telemetri dosyası üretilemedi!", flush=True)
        continue

    print(f" 🎨 [Stage 3.5] V40 SOTA Render Stüdyosu...", flush=True)
    tracking_data = {}
    with open(final_telemetry_path, 'r') as f:
        for line in f:
            parts = line.strip().replace(',', ' ').split()
            if len(parts) < 6: continue
            frame_id, track_id = int(float(parts[0])), int(float(parts[1]))
            if frame_id not in tracking_data: tracking_data[frame_id] = []
            tracking_data[frame_id].append((track_id, float(parts[2]), float(parts[3]), float(parts[4]), float(parts[5])))

    cap = cv2.VideoCapture(temp_source)
    local_rendered = os.path.join(LOCAL_DIR, f"rendered_{seq_name}.mp4")
    writer = cv2.VideoWriter(local_rendered, cv2.VideoWriter_fourcc(*'mp4v'), seq_fps, (int(cap.get(3)), int(cap.get(4))))
    np.random.seed(42); colors = {}
    f_idx = 1
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        if f_idx in tracking_data:
            for tid, l, t, wd, ht in tracking_data[f_idx]:
                x1, y1, x2, y2 = int(l), int(t), int(l + wd), int(t + ht)
                if tid not in colors: colors[tid] = (int(np.random.randint(50, 255)), int(np.random.randint(50, 255)), int(np.random.randint(50, 255)))
                cv2.rectangle(frame, (x1, y1), (x2, y2), colors[tid], 2)
                label = f"ID: {tid}"
                (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
                cv2.rectangle(frame, (x1, y1 - 18), (x1 + tw + 4, y1), colors[tid], -1)
                cv2.putText(frame, label, (x1 + 2, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
        writer.write(frame); f_idx += 1
    cap.release(); writer.release()
    safe_drive_mirror(local_rendered, D_STAGE3, f"Final Tracked ({seq_name})")
    aggressive_ram_purge()

os.chdir("/content")

🛡️ [Ultimate Enterprise Pipeline V40] Sıfır Piksel Zırhı, Fast-ReID ve ECC Aktif...
 🧹 Eski hatalı kurulum tamamen siliniyor...
 📥 HybridSORT orijinal reposu klonlanıyor...
 🛠️ Zırhlı PyTorch ve Numpy 2.x Yamaları uygulanıyor...
 🛡️ Fast-ReID motoruna 'Sıfır Piksel (Empty Array)' zırhı entegre ediliyor...
 ✅ Fast-ReID motoru artık 'Hayalet Kutularda' bile çökmeyecek!
 🔬 demo_track.py otonom RegEx motoruyla zırhlanıyor...

🧠 Hiperparametreler Resmi SOTA ve ECC Kapasitesine Yükseltiliyor...
 ✅ Track_Thresh: 0.6 | NMS_Thresh: 0.7 | Alpha: 0.8 | ReID: True olarak kilitlendi!

══════════════════════════════════════════════════════════════════════
🎯 İŞLENEN SEKANS: MOT17-02-FRCNN (TRAIN SET) [V40 GOD MODE SOTA]
══════════════════════════════════════════════════════════════════════
 ⏭️ [Stage 1] DeepRFT Netleştirilmiş video bulundu, saniyeler içinde atlanıyor.
 🚀 [Stage 3] Hybrid-SORT-ReID (V40 SOTA) Doğrudan Stage 1 Üzerinde Çalışıyor...
2026-08-12 16:24:22.490 | INFO     | __main__:imageflo

In [10]:
import os
import sys
import subprocess
import shutil
import glob
import re
import zipfile
from datetime import datetime
from google.colab import files

file_path = "/content/HybridSORT/tools/demo_track.py"

# =============================================================================
# 1. KUSURSUZ NONETYPE (BOŞ KARE) ZIRHLAMASI
# =============================================================================
with open(file_path, "r", encoding="utf-8") as f:
    code = f.read()

# Hatalı yamanın düzeltilmesi
code = code.replace("return None, None", "return outputs, img_info")

# imageflow_demo içindeki boş kutu kontrolünü güçlendir
old_check = "if outputs[0] is None:"
new_check = "if outputs is None or outputs[0] is None or len(outputs[0]) == 0:"
if new_check not in code:
    code = code.replace(old_check, new_check)

with open(file_path, "w", encoding="utf-8") as f:
    f.write(code)

print("✅ demo_track.py 'NoneType' hatasına karşı %100 zırhlandı!\n")

# =============================================================================
# 2. MOT17-12-FRCNN İÇİN V40 SOTA MOTORUNUN ÇALIŞTIRILMASI
# =============================================================================
seq_name = "MOT17-12-FRCNN"
print(f"🚀 {seq_name} için V40 SOTA Motoru Başlatılıyor (Loglar Açık)...")

cmd = [
    sys.executable, "tools/demo_track.py", "--demo_type", "video",
    "-f", "exps/example/mot/yolox_x_mix_det_hybrid_sort.py", "-c", "pretrained/ocsort_x_mot17.pth.tar",
    "--path", f"/content/local_processing/stage1_deblurred_{seq_name}.mp4",
    "--output_dir", f"/content/sota_run_out_{seq_name}", "--device", "gpu", "--fp16", "--fuse", "--save_result",
    "--ECC", "--hybrid_sort_with_reid", "--with_fastreid",
    "--fast_reid_config", "fast_reid/configs/MOT17/sbs_S50.yml", "--fast_reid_weights", "pretrained/mot17_sbs_S50.pth"
]

os.chdir("/content/HybridSORT")
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

log_history = []
for line in process.stdout:
    log_history.append(line)
    # Konsolu kalabalıklaştırmamak için sadece Processing frame yazılarını göster, ama hatayı kaydet
    if "Processing frame" in line or "save results to" in line:
        sys.stdout.write(line)
        sys.stdout.flush()

process.wait()

if process.returncode != 0:
    print(f"\n❌ [KRİTİK HATA] Sistem yine çöktü! İşte arka planda saklanan hata detayı:\n{''.join(log_history[-30:])}")
    sys.exit(1)

print(f"\n✅ {seq_name} başarıyla işlendi!")

# Telemetriyi kopyala
txt_files = glob.glob(f"/content/sota_run_out_{seq_name}/**/track_vis/*.txt", recursive=True) + glob.glob("/content/HybridSORT/YOLOX_outputs/**/track_vis/*.txt", recursive=True)
if txt_files:
    latest_txt = max(txt_files, key=os.path.getmtime)
    target_path = f"/content/drive/MyDrive/Spikedge_Staj/Tracking/telemetry_test_reid/{seq_name}.txt"
    shutil.copy(latest_txt, target_path)
    print(f"💾 Telemetri Drive'a kilitlendi: {seq_name}.txt\n")

# =============================================================================
# 3. BATCH EVALUATION VE KUSURSUZ ZİPLEME
# =============================================================================
os.chdir("/content")
print("📊 [Batch Evaluation Engine] TrackEval Başlatılıyor...\n")
trackeval_path = "/content/TrackEval"
tracker_name = "HybridSORT_V40_SOTA"
data_dir = os.path.join(trackeval_path, "data/trackers/mot_challenge/MOT17-train", tracker_name, "data")
os.makedirs(data_dir, exist_ok=True)

# Train seti kopyalama (sadece evaluation için)
sequences = ["MOT17-02-FRCNN", "MOT17-04-FRCNN", "MOT17-05-FRCNN", "MOT17-09-FRCNN", "MOT17-10-FRCNN", "MOT17-11-FRCNN", "MOT17-13-FRCNN"]
for seq in sequences:
    src = f"/content/drive/MyDrive/Spikedge_Staj/Tracking/telemetry_train_reid/{seq}.txt"
    dest = os.path.join(data_dir, f"{seq}.txt")
    if os.path.exists(src):
        with open(src, 'r') as f: lines = f.readlines()
        clean_lines = []
        for line in lines:
            parts = line.strip().split(',') if ',' in line else line.strip().split()
            if len(parts) >= 2:
                try:
                    frm = int(float(parts[0])) + 1
                    parts[0] = str(frm)
                    clean_lines.append(",".join(parts))
                except: pass
        with open(dest, 'w') as f: f.write("\n".join(clean_lines) + "\n")

eval_cmd = [sys.executable, os.path.join(trackeval_path, "scripts/run_mot_challenge.py"), "--BENCHMARK", "MOT17", "--SPLIT_TO_EVAL", "train", "--TRACKERS_TO_EVAL", tracker_name, "--METRICS", "HOTA", "CLEAR", "Identity", "--USE_PARALLEL", "False", "--PRINT_RESULTS", "False"]
subprocess.run(eval_cmd, cwd=trackeval_path, capture_output=True, text=True)

summary_file = os.path.join(trackeval_path, "data/trackers/mot_challenge/MOT17-train", tracker_name, "pedestrian_summary.txt")
if os.path.exists(summary_file):
    with open(summary_file, 'r') as f:
        lines = f.readlines()
        if len(lines) >= 2:
            val_dict = dict(zip(lines[0].strip().split(), lines[1].strip().split()))
            print("=====================================================================================")
            print(" 🏆 V40 GOD MODE (RE-ID + ECC) OPTİMİZASYONU SONUÇLARI (TRAIN SETİ)")
            print("=====================================================================================")
            print(f"    ➤ HOTA (Genel Başarı)     : {val_dict.get('HOTA', 'N/A')} %")
            print(f"    ➤ MOTA (Takip Doğruluğu)  : {val_dict.get('MOTA', 'N/A')} %")
            print(f"    ➤ IDF1 (Kimlik Koruma)    : {val_dict.get('IDF1', 'N/A')} %")
            print("=====================================================================================\n")

print("📦 [Format Fixer] İstisnasız +1 Frame Kaydırma ve Zipleme yapılıyor...")
PROJECT_ROOT = "/content/drive/MyDrive/Spikedge_Staj/Tracking"
TEMP_DIR = "/content/Codabench_Temp_42_FIXED"
if os.path.exists(TEMP_DIR): shutil.rmtree(TEMP_DIR)
os.makedirs(TEMP_DIR, exist_ok=True)

all_txt = glob.glob(f"{PROJECT_ROOT}/telemetry_train_reid/*.txt") + glob.glob(f"{PROJECT_ROOT}/telemetry_test_reid/*.txt")
for txt in all_txt:
    with open(txt, 'r') as f: lines = f.readlines()
    clean = []
    seen = set()
    for line in lines:
        p = line.strip().split(',') if ',' in line else line.strip().split()
        if len(p) >= 2:
            try:
                f_id, t_id = int(float(p[0])) + 1, int(float(p[1]))
                if (f_id, t_id) not in seen:
                    seen.add((f_id, t_id))
                    p[0], p[1] = str(f_id), str(t_id)
                    clean.append(",".join(p))
            except: pass
    b_name = os.path.basename(txt)
    with open(os.path.join(TEMP_DIR, b_name), 'w') as f: f.write("\n".join(clean) + "\n")
    with open(os.path.join(TEMP_DIR, b_name.replace("FRCNN", "DPM")), 'w') as f: f.write("\n".join(clean) + "\n")
    with open(os.path.join(TEMP_DIR, b_name.replace("FRCNN", "SDP")), 'w') as f: f.write("\n".join(clean) + "\n")

zip_name = f"Codabench_V40_GOD_MODE_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"
zip_local = os.path.join("/content", zip_name)
with zipfile.ZipFile(zip_local, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in glob.glob(os.path.join(TEMP_DIR, "*.txt")): zf.write(f, os.path.basename(f))
shutil.copy(zip_local, os.path.join(PROJECT_ROOT, zip_name))

print("=======================================================================")
print(f" ✅ GÖREV TAMAMLANDI! {len(glob.glob(os.path.join(TEMP_DIR, '*.txt')))}/42 dosya paketlendi.")
print(f" 💾 ZIP DOSYASI: {zip_name}")
print("=======================================================================")
files.download(zip_local)

✅ demo_track.py 'NoneType' hatasına karşı %100 zırhlandı!

🚀 MOT17-12-FRCNN için V40 SOTA Motoru Başlatılıyor (Loglar Açık)...
2026-08-12 18:27:30.969 | INFO     | __main__:imageflow_demo:260 - Processing frame 0 (100000.00 fps)
2026-08-12 18:27:38.606 | INFO     | __main__:imageflow_demo:260 - Processing frame 20 (3.54 fps)
2026-08-12 18:27:43.536 | INFO     | __main__:imageflow_demo:260 - Processing frame 40 (4.48 fps)
2026-08-12 18:27:49.437 | INFO     | __main__:imageflow_demo:260 - Processing frame 60 (4.76 fps)
2026-08-12 18:27:54.240 | INFO     | __main__:imageflow_demo:260 - Processing frame 80 (5.07 fps)
2026-08-12 18:27:59.329 | INFO     | __main__:imageflow_demo:260 - Processing frame 100 (5.25 fps)
2026-08-12 18:28:04.971 | INFO     | __main__:imageflow_demo:260 - Processing frame 120 (5.31 fps)
2026-08-12 18:28:11.462 | INFO     | __main__:imageflow_demo:260 - Processing frame 140 (5.18 fps)
2026-08-12 18:28:18.638 | INFO     | __main__:imageflow_demo:260 - Processing fram

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
import os, sys, subprocess, shutil, glob, re, zipfile
from datetime import datetime
from google.colab import drive, files

print("🚨 [CRITICAL ALERT] Google Drive FUSE Cache Tuzağı Kırılıyor...")
# 🎯 ASIL SİHİR BURADA: Drive önbelleği zorla temizleniyor!
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)
print("✅ Drive önbelleği temizlendi! En güncel V40 dosyaları buluttan çekiliyor.\n")

print("📊 [Batch Evaluation Engine] V40 Gerçek Metrikler İçin Yeniden Başlatılıyor...")
trackeval_path = "/content/TrackEval"
if os.path.exists(trackeval_path):
    shutil.rmtree(trackeval_path) # Eski kalıntıları tamamen sil
subprocess.run(["git", "clone", "https://github.com/JonathonLuiten/TrackEval.git", trackeval_path])

# NumPy yamaları
for py_file in glob.glob(os.path.join(trackeval_path, "**/*.py"), recursive=True):
    try:
        with open(py_file, 'r', encoding='utf-8') as f: code = f.read()
        code = re.sub(r'\bnp\.float\b', 'float', code)
        code = re.sub(r'\bnp\.int\b', 'int', code)
        code = re.sub(r'\bnp\.bool\b', 'bool', code)
        code = re.sub(r'\bnp\.object\b', 'object', code)
        with open(py_file, 'w', encoding='utf-8') as f: f.write(code)
    except Exception: pass

sequences = ["MOT17-02-FRCNN", "MOT17-04-FRCNN", "MOT17-05-FRCNN", "MOT17-09-FRCNN", "MOT17-10-FRCNN", "MOT17-11-FRCNN", "MOT17-13-FRCNN"]
tracker_name = "HybridSORT_V40_SOTA_REAL"
data_dir = os.path.join(trackeval_path, "data/trackers/mot_challenge/MOT17-train", tracker_name, "data")
os.makedirs(data_dir, exist_ok=True)

# Ground Truth Bağlantısı
drive_gt_dir = "/content/drive/MyDrive/Spikedge_Staj/Tracking/MOT17_Dataset/train"
trackeval_gt_dir = os.path.join(trackeval_path, "data/gt/mot_challenge/MOT17-train")
if os.path.exists(drive_gt_dir) and not os.path.exists(trackeval_gt_dir):
    os.makedirs(os.path.dirname(trackeval_gt_dir), exist_ok=True)
    os.symlink(drive_gt_dir, trackeval_gt_dir)

DRIVE_TELEMETRY = "/content/drive/MyDrive/Spikedge_Staj/Tracking/telemetry_train_reid"
for seq in sequences:
    src_txt = os.path.join(DRIVE_TELEMETRY, f"{seq}.txt")
    dest_txt = os.path.join(data_dir, f"{seq}.txt")
    if os.path.exists(src_txt):
        with open(src_txt, 'r') as f: lines = f.readlines()
        clean_lines = []
        seen = set()
        for line in lines:
            parts = line.strip().split(',') if ',' in line else line.strip().split()
            if len(parts) >= 2:
                try:
                    frm, tid = int(float(parts[0])) + 1, int(float(parts[1]))
                    if (frm, tid) not in seen:
                        seen.add((frm, tid))
                        parts[0], parts[1] = str(frm), str(tid)
                        clean_lines.append(",".join(parts))
                except: pass
        with open(dest_txt, 'w') as f: f.write("\n".join(clean_lines) + "\n")

seqmap_dir = os.path.join(trackeval_path, "data/gt/mot_challenge/seqmaps")
os.makedirs(seqmap_dir, exist_ok=True)
with open(os.path.join(seqmap_dir, "MOT17-train.txt"), "w") as f:
    f.write("name\n")
    for seq in sequences: f.write(f"{seq}\n")

print("🚀 TrackEval Motoru GERÇEK V40 Analizini Başlattı...\n")
eval_cmd = [sys.executable, os.path.join(trackeval_path, "scripts/run_mot_challenge.py"), "--BENCHMARK", "MOT17", "--SPLIT_TO_EVAL", "train", "--TRACKERS_TO_EVAL", tracker_name, "--METRICS", "HOTA", "CLEAR", "Identity", "--USE_PARALLEL", "False", "--PRINT_RESULTS", "False"]
subprocess.run(eval_cmd, cwd=trackeval_path, capture_output=True, text=True)

summary_file = os.path.join(trackeval_path, "data/trackers/mot_challenge/MOT17-train", tracker_name, "pedestrian_summary.txt")
if os.path.exists(summary_file):
    with open(summary_file, 'r') as f:
        lines = f.readlines()
        if len(lines) >= 2:
            val_dict = dict(zip(lines[0].strip().split(), lines[1].strip().split()))
            print("=====================================================================================")
            print(" 🏆 GERÇEK V40 GOD MODE (FAST-REID + ECC) SONUÇLARI (TRAIN SETİ)")
            print("=====================================================================================")
            print(f"    ➤ HOTA (Genel Başarı)     : {val_dict.get('HOTA', 'N/A')} %")
            print(f"    ➤ MOTA (Takip Doğruluğu)  : {val_dict.get('MOTA', 'N/A')} %")
            print(f"    ➤ IDF1 (Kimlik Koruma)    : {val_dict.get('IDF1', 'N/A')} %")
            print(f"    ➤ MOTP (Kutu Hassasiyeti) : {val_dict.get('MOTP', 'N/A')} %")
            print(f"    ➤ IDSW (Kimlik Değişimi)  : {val_dict.get('IDSW', 'N/A')}")
            print("=====================================================================================\n")

print("📦 [Format Fixer] Gerçek V40 Dosyaları Paketleniyor...")
PROJECT_ROOT = "/content/drive/MyDrive/Spikedge_Staj/Tracking"
TEMP_DIR = "/content/Codabench_Temp_42_REAL_V40"
if os.path.exists(TEMP_DIR): shutil.rmtree(TEMP_DIR)
os.makedirs(TEMP_DIR, exist_ok=True)

all_txt = glob.glob(f"{PROJECT_ROOT}/telemetry_train_reid/*.txt") + glob.glob(f"{PROJECT_ROOT}/telemetry_test_reid/*.txt")
for txt in all_txt:
    with open(txt, 'r') as f: lines = f.readlines()
    clean = []
    seen = set()
    for line in lines:
        p = line.strip().split(',') if ',' in line else line.strip().split()
        if len(p) >= 2:
            try:
                f_id, t_id = int(float(p[0])) + 1, int(float(p[1]))
                if (f_id, t_id) not in seen:
                    seen.add((f_id, t_id))
                    p[0], p[1] = str(f_id), str(t_id)
                    clean.append(",".join(p))
            except: pass
    b_name = os.path.basename(txt)
    with open(os.path.join(TEMP_DIR, b_name), 'w') as f: f.write("\n".join(clean) + "\n")
    with open(os.path.join(TEMP_DIR, b_name.replace("FRCNN", "DPM")), 'w') as f: f.write("\n".join(clean) + "\n")
    with open(os.path.join(TEMP_DIR, b_name.replace("FRCNN", "SDP")), 'w') as f: f.write("\n".join(clean) + "\n")

zip_name = f"Codabench_V40_REAL_GOD_MODE_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"
zip_local = os.path.join("/content", zip_name)
with zipfile.ZipFile(zip_local, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in glob.glob(os.path.join(TEMP_DIR, "*.txt")): zf.write(f, os.path.basename(f))
shutil.copy(zip_local, os.path.join(PROJECT_ROOT, zip_name))

print("=======================================================================")
print(f" ✅ ZİP HAZIR! Toplam {len(glob.glob(os.path.join(TEMP_DIR, '*.txt')))} dosya paketlendi.")
print(f" 💾 İNDİRİLECEK DOSYA: {zip_name}")
print("=======================================================================")
files.download(zip_local)

🚨 [CRITICAL ALERT] Google Drive FUSE Cache Tuzağı Kırılıyor...
Mounted at /content/drive
✅ Drive önbelleği temizlendi! En güncel V40 dosyaları buluttan çekiliyor.

📊 [Batch Evaluation Engine] V40 Gerçek Metrikler İçin Yeniden Başlatılıyor...
🚀 TrackEval Motoru GERÇEK V40 Analizini Başlattı...

 🏆 GERÇEK V40 GOD MODE (FAST-REID + ECC) SONUÇLARI (TRAIN SETİ)
    ➤ HOTA (Genel Başarı)     : 74.621 %
    ➤ MOTA (Takip Doğruluğu)  : 88.907 %
    ➤ IDF1 (Kimlik Koruma)    : 81.883 %
    ➤ MOTP (Kutu Hassasiyeti) : 88.209 %
    ➤ IDSW (Kimlik Değişimi)  : 598

📦 [Format Fixer] Gerçek V40 Dosyaları Paketleniyor...
 ✅ ZİP HAZIR! Toplam 42 dosya paketlendi.
 💾 İNDİRİLECEK DOSYA: Codabench_V40_REAL_GOD_MODE_20260812_184708.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ==============================================================================
# 🎬 PIPELINE VİZYON MOTORU: 2x2 GRID (YAN YANA GÖRSELLEŞTİRME)
# ==============================================================================
import os
import subprocess
from IPython.display import HTML
from base64 import b64encode

print("🎬 [Vision Engine] 2x2 Grid Render Motoru Başlatılıyor...\n")

# ⚠️ BURAYI KENDİ DRIVE YOLLARINA GÖRE GÜNCELLE (Örnek olarak MOT17-04 seçilmiştir)
# Dosyaların tam nerede olduğunu Drive'dan kopyala/yapıştır.
vid_input = "/content/drive/MyDrive/Spikedge_Staj/Tracking/input_videos_test/MOT17-08-FRCNN_raw_input.mp4"
vid_stage1 = "/content/drive/MyDrive/Spikedge_Staj/Tracking/intermediate_test/stage1_deblurred_MOT17-08-FRCNN.mp4"
vid_stage2 = "/content/drive/MyDrive/Spikedge_Staj/Tracking/intermediate_test/stage2_stabilized_MOT17-08-FRCNN.mp4"
vid_final = "/content/drive/MyDrive/Spikedge_Staj/Tracking/output_tracks_test/final_tracked_MOT17-08-FRCNN.mp4"

output_grid = "/content/drive/MyDrive/Spikedge_Staj/Tracking/Dashboard_MOT17-04_Grid.mp4"

# Dosya kontrolü
for v, name in zip([vid_input, vid_stage1, vid_stage2, vid_final], ["Input", "Stage1", "Stage2", "Final"]):
    if not os.path.exists(v):
        print(f" ❌ HATA: {name} videosu bulunamadı! Yol: {v}")
    else:
        print(f" ✔️ {name} doğrulandı.")

print("\n ⚙️ FFmpeg Complex Filter Ağacı Kuruluyor (Bu işlem 3-5 dakika sürebilir)...")

# FFmpeg Complex Filter Komutu
# Her videoyu 960x540'a küçültüyor, üzerlerine isimlerini yazıyor ve 2x2 birleştiriyor.
ffmpeg_cmd = [
    "ffmpeg", "-y",
    "-i", vid_input,
    "-i", vid_stage1,
    "-i", vid_stage2,
    "-i", vid_final,
    "-filter_complex",
    """
    [0:v]scale=960:540,drawtext=text='1. RAW INPUT':fontcolor=white:fontsize=36:box=1:boxcolor=black@0.6:x=20:y=20[v0];
    [1:v]scale=960:540,drawtext=text='2. STAGE 1 (DEBLURRING)':fontcolor=white:fontsize=36:box=1:boxcolor=black@0.6:x=20:y=20[v1];
    [2:v]scale=960:540,drawtext=text='3. STAGE 2 (STABILIZATION)':fontcolor=white:fontsize=36:box=1:boxcolor=black@0.6:x=20:y=20[v2];
    [3:v]scale=960:540,drawtext=text='4. FINAL OUTPUT (HybridSORT)':fontcolor=white:fontsize=36:box=1:boxcolor=black@0.6:x=20:y=20[v3];
    [v0][v1]hstack=inputs=2[top];
    [v2][v3]hstack=inputs=2[bottom];
    [top][bottom]vstack=inputs=2[out]
    """,
    "-map", "[out]",
    "-c:v", "libx264", "-crf", "23", "-preset", "fast",
    output_grid
]

# Render İşlemi
try:
    subprocess.run(ffmpeg_cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print(" ✅ Render Başarılı! 2x2 Grid Video Drive'a Kaydedildi.")
except subprocess.CalledProcessError as e:
    print(" ❌ Render Hatası! Hata detayı:\n", e.stderr.decode('utf-8'))

# ==============================================================================
# 📺 COLAB İÇİNDE OYNATMA (HTML5)
# ==============================================================================
print(" 📺 Video Colab Ekranına Yansıtılıyor...")

if os.path.exists(output_grid):
    mp4 = open(output_grid,'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

    display(HTML(f"""
    <video width="100%" controls>
          <source src="{data_url}" type="video/mp4">
    </video>
    """))
else:
    print("⚠️ Video dosyası Colab'e aktarılamadı, lütfen Drive'dan kontrol edin.")